In [ ]:
# Cell 1: Imports
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

In [ ]:
# Cell 2: Resolve repository root and inspect artifacts
def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    candidates = [start, *start.parents]
    markers = ["README.md", "benchmark_methods_gpu.py", "Report_SER.tex"]

    for candidate in candidates:
        if all((candidate / marker).exists() for marker in markers):
            return candidate

    for candidate in candidates:
        if (candidate / "internal" / "SER.ipynb").exists() or (candidate / "split_manifest.json").exists():
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()
print(f"Repository root: {REPO_ROOT}")

print("Key files present:")
for rel in [
    "benchmark_results_gpu.json",
    "split_manifest.json",
    "sample_visec.wav",
    "Report_SER.tex",
    "ECAPA/predict_emotion.py",
    "DFAT_Hybrid_Fusion/predict_dualstream.py",
]:
    print(f"- {rel}: {(REPO_ROOT / rel).exists()}")


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def pretty_print_tree(paths: Iterable[Path], max_items: int = 20) -> None:
    items = [str(p.relative_to(REPO_ROOT)) for p in paths]
    for item in items[:max_items]:
        print(item)
    if len(items) > max_items:
        print(f"... and {len(items) - max_items} more")


print("\nNotebook-relevant files:")
pretty_print_tree(sorted(REPO_ROOT.rglob("*.py"))[:50])

In [ ]:
# Cell 3: Load benchmark and ablation artifacts
benchmark_path = REPO_ROOT / "benchmark_results_gpu.json"
ablation_path = REPO_ROOT / "DFAT_Hybrid_Fusion" / "ablation_results.json"
split_manifest_path = REPO_ROOT / "split_manifest.json"
sample_audio_path = REPO_ROOT / "sample_visec.wav"

benchmark = load_json(benchmark_path) if benchmark_path.exists() else {}
ablation = load_json(ablation_path) if ablation_path.exists() else {}
split_manifest = load_json(split_manifest_path) if split_manifest_path.exists() else {}

print(f"Benchmark keys: {list(benchmark)[:10]}")
print(f"Ablation type: {type(ablation).__name__}")
print(f"Split manifest type: {type(split_manifest).__name__}")
print(f"Sample audio exists: {sample_audio_path.exists()}")

In [ ]:
# Cell 4: Utility helpers
def benchmark_to_dataframe(obj: Any) -> pd.DataFrame:
    if isinstance(obj, dict):
        if "ranked_results" in obj and isinstance(obj["ranked_results"], list):
            return pd.DataFrame(obj["ranked_results"])
        if "results" in obj and isinstance(obj["results"], list):
            return pd.DataFrame(obj["results"])
        if all(isinstance(v, dict) for v in obj.values()):
            rows = []
            for k, v in obj.items():
                row = {"method": k}
                row.update(v)
                rows.append(row)
            return pd.DataFrame(rows)
        return pd.DataFrame([obj])
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    return pd.DataFrame()


def pick_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def run_script(script: Path, args: list[str]) -> tuple[int, str, str]:
    cmd = [sys.executable, str(script), *args]
    proc = subprocess.run(cmd, cwd=str(script.parent), capture_output=True, text=True)
    return proc.returncode, proc.stdout.strip(), proc.stderr.strip()


def extract_json_object(text: str) -> Any | None:
    text = text.strip()
    if not text:
        return None

    try:
        return json.loads(text)
    except Exception:
        pass

    candidates = re.findall(r"\{.*\}", text, flags=re.DOTALL)
    for candidate in reversed(candidates):
        try:
            return json.loads(candidate)
        except Exception:
            continue
    return None

In [ ]:
# Cell 5: Main benchmark table
bench_df = benchmark_to_dataframe(benchmark)

if not bench_df.empty:
    display_cols = [c for c in ["method", "category", "wF1", "mF1", "acc", "latency_ms"] if c in bench_df.columns]
    display(bench_df[display_cols] if display_cols else bench_df)
else:
    print("No benchmark dataframe could be constructed from benchmark_results_gpu.json")

In [ ]:
# Cell 6: Visualize the main benchmark comparison
if not bench_df.empty:
    method_col = pick_col(bench_df, ["method", "model", "name"])
    wf1_col = pick_col(bench_df, ["wF1", "wf1", "weighted_f1", "weighted f1"])
    mF1_col = pick_col(bench_df, ["mF1", "mf1", "macro_f1", "macro f1"])
    acc_col = pick_col(bench_df, ["acc", "accuracy"])

    if method_col and wf1_col:
        plot_df = bench_df[[method_col, wf1_col] + [c for c in [mF1_col, acc_col] if c]].copy()
        plot_df = plot_df.sort_values(by=wf1_col, ascending=False)

        ax = plot_df.set_index(method_col)[wf1_col].plot(kind="bar")
        ax.set_title("Main benchmark: Weighted F1")
        ax.set_xlabel("Method")
        ax.set_ylabel("Weighted F1")
        plt.tight_layout()
        plt.show()

        if mF1_col and acc_col:
            fig, ax = plt.subplots()
            plot_df.set_index(method_col)[[wf1_col, mF1_col, acc_col]].plot(kind="bar", ax=ax)
            ax.set_title("Main benchmark metrics")
            ax.set_xlabel("Method")
            ax.set_ylabel("Score")
            plt.tight_layout()
            plt.show()
    else:
        print("Could not identify method/wF1 columns for plotting.")
else:
    print("Benchmark dataframe is empty, skipping plot.")

In [ ]:
# Cell 7: DFAT ablation study
ablation_df = benchmark_to_dataframe(ablation)

if not ablation_df.empty:
    display(ablation_df)

    label_col = pick_col(ablation_df, ["configuration", "method", "setting", "name"])
    score_col = pick_col(ablation_df, ["ensemble wf1", "wf1", "weighted f1", "ensemble mf1", "mf1", "acc", "accuracy"])

    if label_col and score_col:
        fig, ax = plt.subplots()
        plot_df = ablation_df[[label_col, score_col]].sort_values(score_col, ascending=False)
        plot_df.set_index(label_col)[score_col].plot(kind="bar", ax=ax)
        ax.set_title("DFAT ablation")
        ax.set_xlabel("Configuration")
        ax.set_ylabel(score_col)
        plt.tight_layout()
        plt.show()
else:
    print("No ablation dataframe could be constructed from ablation_results.json")

In [ ]:
# Cell 8: Optional sample inference on sample_visec.wav
def try_predict_ecapa(audio_path: Path) -> Any | None:
    script = REPO_ROOT / "ECAPA" / "predict_emotion.py"
    model_dir = REPO_ROOT / "ECAPA" / "emotion_model"

    if not script.exists():
        print("ECAPA predict script not found.")
        return None

    rc, out, err = run_script(script, [str(audio_path), "--model_dir", str(model_dir)])
    if rc != 0:
        print("ECAPA script failed:")
        print(err or out)
        return None

    parsed = extract_json_object(out)
    print("ECAPA raw output:")
    print(out)
    return parsed if parsed is not None else out


def try_predict_dfat(audio_path: Path) -> Any | None:
    script = REPO_ROOT / "DFAT_Hybrid_Fusion" / "predict_dualstream.py"
    model_dir = REPO_ROOT / "DFAT_Hybrid_Fusion" / "dualstream_model"

    if not script.exists():
        print("DFAT predict script not found.")
        return None

    rc, out, err = run_script(script, ["--audio_file", str(audio_path), "--model_dir", str(model_dir)])
    if rc != 0:
        print("DFAT script failed:")
        print(err or out)
        return None

    parsed = extract_json_object(out)
    print("DFAT raw output:")
    print(out)
    return parsed if parsed is not None else out


if sample_audio_path.exists():
    print(f"Sample audio: {sample_audio_path}")
    ecapa_pred = try_predict_ecapa(sample_audio_path)
    dfat_pred = try_predict_dfat(sample_audio_path)

    if ecapa_pred is not None:
        print("\nECAPA prediction parsed:")
        display(ecapa_pred)

    if dfat_pred is not None:
        print("\nDFAT prediction parsed:")
        display(dfat_pred)
else:
    print("sample_visec.wav is missing; skip demo inference.")

In [ ]:
# Cell 9: Notebook takeaway
print(
    "This notebook is intentionally limited to presentation and inference demonstration. "
    "The thesis-grade experimental logic stays in the dedicated scripts and module structure."
)